# Population-Level Statistics

Aggregate per-light-curve statistics across all candidates that passed through
the malca pipeline cluster stage, broken down **per magnitude bin**.

Data comes from the pre-computed `stats_*` columns in each run's
`results/lc_events_enriched.parquet`.

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

import json as _json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats as sp_stats

# ── pretty defaults ──
sns.set_theme(style='whitegrid', font_scale=1.05)
plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 150,
    'figure.figsize': (10, 5),
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})

BIN_PALETTE = {
    '12–12.5': '#1b9e77',
    '12.5–13': '#d95f02',
    '13–13.5': '#7570b3',
    '13.5–14': '#e7298a',
    '14–14.5': '#66a61e',
    '14.5–15': '#e6ab02',
}
BIN_ORDER = list(BIN_PALETTE.keys())

def ordered_mag_bins(values):
    present = {str(v) for v in values if pd.notna(v)}
    return [label for label in BIN_ORDER if label in present]


def palette_for_bins(values):
    labels = ordered_mag_bins(values)
    return {label: BIN_PALETTE[label] for label in labels}


In [ ]:
# ── configure run directories ────────────────────────────────────
RUN_ROOT_CANDIDATES = [Path('../../output/runs'), Path('output/runs')]
RUNS_ROOT = next((path for path in RUN_ROOT_CANDIDATES if path.exists()), RUN_ROOT_CANDIDATES[0])
print(f'Using run root: {RUNS_ROOT.resolve()}')

RUN_PATTERNS = [
    ('12_12.5', '12–12.5'),
    ('12.5_13', '12.5–13'),
    ('13_13.5', '13–13.5'),
    ('13.5_14', '13.5–14'),
    ('14_14.5', '14–14.5'),
    ('14.5_15', '14.5–15'),
]

# ── load & tag ───────────────────────────────────────────────────
frames = []
for token, label in RUN_PATTERNS:
    matches = sorted(RUNS_ROOT.glob(f'*{token}*/results/lc_events_enriched.parquet'))
    if not matches:
        print(f'⚠  no enriched parquet found for {label} under {RUNS_ROOT}')
        continue

    p = max(matches, key=lambda candidate: candidate.stat().st_mtime)
    if len(matches) > 1:
        print(f'ℹ  {label:10s}  using newest match: {p.parent.parent.name}')

    df = pd.read_parquet(p).copy()
    df['mag_bin'] = label
    df['run_name'] = p.parent.parent.name
    df['candidate_file'] = str(p)
    frames.append(df)
    print(f'✓  {label:10s}  →  {len(df):>6,} candidates  ({p.parent.parent.name})')

if not frames:
    searched = ', '.join(token for token, _ in RUN_PATTERNS)
    raise FileNotFoundError(
        f'No lc_events_enriched.parquet files found under {RUNS_ROOT} for patterns: {searched}'
    )

df_all = pd.concat(frames, ignore_index=True)
print(f'\nTotal candidates: {len(df_all):,}')


In [ ]:
# ── identify stats columns ───────────────────────────────────────
STATS_COLS = sorted([c for c in df_all.columns if c.startswith('stats_')])

# Exclude columns that are all-NaN (e.g. Lomb-Scargle if not computed)
STATS_COLS = [c for c in STATS_COLS if df_all[c].notna().any()]

print(f'{len(STATS_COLS)} usable stats columns')
for c in STATS_COLS:
    print(f'  {c}')

## \u00a72 \u2013 Summary Table
Candidate counts per mag bin and overall.

In [ ]:
summary_rows = []
for label in ordered_mag_bins(df_all['mag_bin']):
    sub = df_all[df_all['mag_bin'] == label]
    row = {
        'mag_bin': label,
        'n_candidates': len(sub),
        'n_passed_all_filters': int((~sub['failed_any']).sum()) if 'failed_any' in sub.columns else len(sub),
    }
    summary_rows.append(row)

summary_rows.append({
    'mag_bin': 'ALL',
    'n_candidates': len(df_all),
    'n_passed_all_filters': int((~df_all['failed_any']).sum()) if 'failed_any' in df_all.columns else len(df_all),
})

pd.DataFrame(summary_rows).style.format({'n_candidates': '{:,}', 'n_passed_all_filters': '{:,}'})


## \u00a73 \u2013 Distributional Statistics per Mag Bin
For every numeric `stats_*` column: count, mean, std, median, MAD, percentiles.

In [ ]:
def descriptive_table(df, col):
    rows = []
    for label in ordered_mag_bins(df['mag_bin']) + ['ALL']:
        s = df[col] if label == 'ALL' else df.loc[df['mag_bin'] == label, col]
        s = s.dropna()
        if s.empty:
            continue
        rows.append({
            'mag_bin': label,
            'count': len(s),
            'mean': s.mean(),
            'std': s.std(),
            'median': s.median(),
            'MAD': np.median(np.abs(s - s.median())),
            'p05': s.quantile(0.05),
            'p25': s.quantile(0.25),
            'p75': s.quantile(0.75),
            'p95': s.quantile(0.95),
            'min': s.min(),
            'max': s.max(),
        })
    return pd.DataFrame(rows).set_index('mag_bin')


In [ ]:
# Print summary for every stats column
for col in STATS_COLS:
    short = col.removeprefix('stats_')
    print(f'\n\u2501\u2501\u2501  {short}  \u2501\u2501\u2501')
    display(descriptive_table(df_all, col))

## \u00a74 \u2013 Histograms / KDE per Mag Bin
Key statistics overlaid by magnitude bin.

In [ ]:
STAT_GROUPS = {
    'Cadence': [
        'stats_cadence_mean_dt_days',
        'stats_cadence_median_dt_days',
        'stats_duty_cycle_fraction',
    ],
    'Photometric scatter': [
        'stats_photometry_std_mag',
        'stats_photometry_robust_sigma_mag',
        'stats_photometry_IQR_mag',
    ],
    'Variability indices': [
        'stats_variability_reduced_chi2_vs_constant',
        'stats_variability_von_neumann_ratio',
        'stats_variability_lag1_autocorr',
        'stats_variability_stetson_J',
        'stats_variability_stetson_K',
    ],
    'Error / SNR': [
        'stats_error_and_snr_stats_error_median',
        'stats_error_and_snr_stats_snr_median',
    ],
    'Trend': [
        'stats_trend_slope_mag_per_year',
        'stats_trend_r2',
    ],
    'Coverage': [
        'stats_time_span_days',
        'stats_n_unique_nights',
        'stats_file_points_kept_after_filter',
    ],
}

In [ ]:
def plot_hist_group(group_name, cols, log_x_cols=None):
    log_x_cols = log_x_cols or set()
    n = len(cols)
    ncols_g = min(n, 3)
    nrows = int(np.ceil(n / ncols_g))
    fig, axes = plt.subplots(nrows, ncols_g,
                             figsize=(5 * ncols_g, 4 * nrows), squeeze=False)
    fig.suptitle(group_name, fontsize=15, fontweight='bold', y=1.02)

    for idx, col in enumerate(cols):
        ax = axes[idx // ncols_g, idx % ncols_g]
        short = col.removeprefix('stats_')

        for label, color in BIN_PALETTE.items():
            s = df_all.loc[df_all['mag_bin'] == label, col].dropna()
            if s.empty:
                continue
            data = s.values
            use_log = col in log_x_cols
            if use_log:
                data = data[data > 0]
                data = np.log10(data)
                xlabel = f'log10({short})'
            else:
                xlabel = short

            lo, hi = np.percentile(data, [1, 99])
            data_clip = data[(data >= lo) & (data <= hi)]
            ax.hist(data_clip, bins=50, alpha=0.35, color=color,
                    label=label, density=True)
            try:
                kde = sp_stats.gaussian_kde(data_clip)
                xs = np.linspace(data_clip.min(), data_clip.max(), 200)
                ax.plot(xs, kde(xs), color=color, lw=1.8)
            except Exception:
                pass

        ax.set_xlabel(xlabel, fontsize=9)
        ax.set_ylabel('density')
        ax.legend(fontsize=8)
        ax.tick_params(labelsize=8)

    for idx in range(n, nrows * ncols_g):
        axes[idx // ncols_g, idx % ncols_g].set_visible(False)

    fig.tight_layout()
    plt.show()

In [ ]:
LOG_X = {
    'stats_variability_reduced_chi2_vs_constant',
    'stats_variability_stetson_I',
}

for group_name, cols in STAT_GROUPS.items():
    cols = [c for c in cols if c in STATS_COLS]
    if cols:
        plot_hist_group(group_name, cols, log_x_cols=LOG_X)

## \u00a75 \u2013 Correlation Heatmap
Pearson correlation of key scalar stats, per mag bin and combined.

In [ ]:
KEY_STATS = [
    'stats_photometry_std_mag',
    'stats_photometry_robust_sigma_mag',
    'stats_photometry_IQR_mag',
    'stats_variability_reduced_chi2_vs_constant',
    'stats_variability_von_neumann_ratio',
    'stats_variability_lag1_autocorr',
    'stats_variability_stetson_J',
    'stats_variability_stetson_K',
    'stats_cadence_median_dt_days',
    'stats_duty_cycle_fraction',
    'stats_error_and_snr_stats_snr_median',
    'stats_trend_slope_mag_per_year',
    'stats_trend_r2',
    'stats_time_span_days',
    'stats_file_points_kept_after_filter',
]
KEY_STATS = [c for c in KEY_STATS if c in STATS_COLS]
short_labels = [c.removeprefix('stats_') for c in KEY_STATS]

In [ ]:
labels_to_plot = ordered_mag_bins(df_all['mag_bin']) + ['ALL']
n_panels = len(labels_to_plot)
fig, axes = plt.subplots(1, n_panels, figsize=(7 * n_panels, 6), squeeze=False)

for i, label in enumerate(labels_to_plot):
    ax = axes[0, i]
    sub = df_all if label == 'ALL' else df_all[df_all['mag_bin'] == label]
    corr = sub[KEY_STATS].corr()
    corr.index = short_labels
    corr.columns = short_labels
    sns.heatmap(corr, ax=ax, cmap='RdBu_r', center=0, vmin=-1, vmax=1,
                square=True, linewidths=0.3,
                cbar_kws={'shrink': 0.7},
                annot=True, fmt='.2f', annot_kws={'size': 6})
    ax.set_title(f'mag bin {label}' if label != 'ALL' else 'ALL combined',
                 fontsize=12)
    ax.tick_params(labelsize=7, rotation=0)
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')

fig.tight_layout()
plt.show()


## \u00a76 \u2013 Mag-Bin Comparison: Violin Plots
Side-by-side violin plots of key stats grouped by magnitude bin.

In [ ]:
VIOLIN_STATS = [
    'stats_photometry_robust_sigma_mag',
    'stats_variability_reduced_chi2_vs_constant',
    'stats_variability_von_neumann_ratio',
    'stats_variability_stetson_J',
    'stats_variability_stetson_K',
    'stats_cadence_median_dt_days',
    'stats_duty_cycle_fraction',
    'stats_error_and_snr_stats_snr_median',
    'stats_trend_r2',
    'stats_file_points_kept_after_filter',
]
VIOLIN_STATS = [c for c in VIOLIN_STATS if c in STATS_COLS]

LOG_VIOLIN = {
    'stats_variability_reduced_chi2_vs_constant',
    'stats_file_points_kept_after_filter',
}

bin_order = ordered_mag_bins(df_all['mag_bin'])
palette = [BIN_PALETTE[label] for label in bin_order]

n = len(VIOLIN_STATS)
ncols_v = 3
nrows = int(np.ceil(n / ncols_v))
fig, axes = plt.subplots(nrows, ncols_v,
                         figsize=(5 * ncols_v, 4.5 * nrows), squeeze=False)

for idx, col in enumerate(VIOLIN_STATS):
    ax = axes[idx // ncols_v, idx % ncols_v]
    short = col.removeprefix('stats_')
    use_log = col in LOG_VIOLIN

    plot_df = df_all[['mag_bin', col]].dropna()
    if use_log:
        plot_df = plot_df[plot_df[col] > 0].copy()
        plot_df[col] = np.log10(plot_df[col])
        ylabel = f'log10({short})'
    else:
        ylabel = short

    lo, hi = plot_df[col].quantile([0.01, 0.99])
    plot_df = plot_df[(plot_df[col] >= lo) & (plot_df[col] <= hi)]

    sns.violinplot(data=plot_df, x='mag_bin', y=col, ax=ax,
                   palette=palette, inner='quartile', linewidth=0.8,
                   order=bin_order, cut=0, density_norm='width')
    ax.set_ylabel(ylabel, fontsize=9)
    ax.set_xlabel('')
    ax.set_title(short, fontsize=10, fontweight='bold')
    ax.tick_params(labelsize=8)

for idx in range(n, nrows * ncols_v):
    axes[idx // ncols_v, idx % ncols_v].set_visible(False)

fig.suptitle('Distribution comparisons across mag bins',
             fontsize=14, fontweight='bold', y=1.01)
fig.tight_layout()
plt.show()


## \u00a78 \u2013 Scatter Plots
Pairwise scatter plots of key statistics, colored by magnitude bin.
Points are drawn with transparency to reveal density structure.

In [ ]:
def scatter_grid(pairs, title, log_axes=None, figsize_per=4.5):
    """Draw a grid of scatter plots, one per (x, y) pair."""
    log_axes = log_axes or {}
    n = len(pairs)
    ncols_s = min(n, 3)
    nrows = int(np.ceil(n / ncols_s))
    fig, axes = plt.subplots(nrows, ncols_s,
                             figsize=(figsize_per * ncols_s, figsize_per * nrows),
                             squeeze=False)
    fig.suptitle(title, fontsize=15, fontweight='bold', y=1.02)

    for idx, (xcol, ycol) in enumerate(pairs):
        ax = axes[idx // ncols_s, idx % ncols_s]
        for label, color in BIN_PALETTE.items():
            mask = df_all['mag_bin'] == label
            x = df_all.loc[mask, xcol].values.copy()
            y = df_all.loc[mask, ycol].values.copy()
            valid = np.isfinite(x) & np.isfinite(y)
            if xcol in log_axes:
                valid &= (x > 0)
            if ycol in log_axes:
                valid &= (y > 0)
            x, y = x[valid], y[valid]
            if xcol in log_axes:
                x = np.log10(x)
            if ycol in log_axes:
                y = np.log10(y)
            ax.scatter(x, y, s=6, alpha=0.25, color=color, label=label,
                       edgecolors='none', rasterized=True)

        xshort = xcol.removeprefix('stats_')
        yshort = ycol.removeprefix('stats_')
        if xcol in log_axes:
            xshort = f'log10({xshort})'
        if ycol in log_axes:
            yshort = f'log10({yshort})'
        ax.set_xlabel(xshort, fontsize=8)
        ax.set_ylabel(yshort, fontsize=8)
        ax.tick_params(labelsize=7)
        ax.legend(fontsize=7, markerscale=3)

    for idx in range(n, nrows * ncols_s):
        axes[idx // ncols_s, idx % ncols_s].set_visible(False)
    fig.tight_layout()
    plt.show()

### 8a \u2013 Variability Diagnostics
Scatter plots pairing the core variability indices against each other.

In [ ]:
scatter_grid([
    ('stats_variability_reduced_chi2_vs_constant', 'stats_variability_von_neumann_ratio'),
    ('stats_variability_reduced_chi2_vs_constant', 'stats_variability_stetson_J'),
    ('stats_variability_reduced_chi2_vs_constant', 'stats_variability_lag1_autocorr'),
    ('stats_variability_von_neumann_ratio',        'stats_variability_lag1_autocorr'),
    ('stats_variability_stetson_J',                'stats_variability_stetson_K'),
    ('stats_variability_stetson_J',                'stats_variability_von_neumann_ratio'),
], title='Variability Diagnostics',
   log_axes={'stats_variability_reduced_chi2_vs_constant'})

### 8b \u2013 Photometric Scatter vs Variability
How photometric dispersion relates to variability metrics.

In [ ]:
scatter_grid([
    ('stats_photometry_robust_sigma_mag', 'stats_variability_reduced_chi2_vs_constant'),
    ('stats_photometry_robust_sigma_mag', 'stats_variability_stetson_J'),
    ('stats_photometry_IQR_mag',          'stats_variability_von_neumann_ratio'),
    ('stats_photometry_std_mag',          'stats_variability_lag1_autocorr'),
    ('stats_photometry_robust_sigma_mag', 'stats_variability_stetson_K'),
    ('stats_photometry_IQR_mag',          'stats_variability_stetson_J'),
], title='Photometric Scatter vs Variability',
   log_axes={'stats_variability_reduced_chi2_vs_constant'})

### 8c \u2013 Cadence & Coverage vs Variability
Whether cadence or coverage properties bias the variability diagnostics.

In [ ]:
scatter_grid([
    ('stats_cadence_median_dt_days',         'stats_variability_reduced_chi2_vs_constant'),
    ('stats_duty_cycle_fraction',            'stats_variability_reduced_chi2_vs_constant'),
    ('stats_file_points_kept_after_filter',  'stats_variability_reduced_chi2_vs_constant'),
    ('stats_cadence_median_dt_days',         'stats_variability_von_neumann_ratio'),
    ('stats_file_points_kept_after_filter',  'stats_variability_stetson_J'),
    ('stats_time_span_days',                 'stats_variability_stetson_J'),
], title='Cadence & Coverage vs Variability',
   log_axes={'stats_variability_reduced_chi2_vs_constant',
             'stats_file_points_kept_after_filter'})

### 8d \u2013 Error / SNR vs Photometric Properties
How photometric errors and signal-to-noise relate to scatter and variability.

In [ ]:
scatter_grid([
    ('stats_error_and_snr_stats_snr_median',   'stats_photometry_robust_sigma_mag'),
    ('stats_error_and_snr_stats_error_median', 'stats_photometry_std_mag'),
    ('stats_error_and_snr_stats_snr_median',   'stats_variability_reduced_chi2_vs_constant'),
    ('stats_error_and_snr_stats_snr_median',   'stats_variability_stetson_J'),
    ('stats_error_and_snr_stats_error_median', 'stats_variability_von_neumann_ratio'),
    ('stats_error_and_snr_stats_snr_median',   'stats_duty_cycle_fraction'),
], title='Error / SNR vs Photometry & Variability',
   log_axes={'stats_variability_reduced_chi2_vs_constant'})

### 8e \u2013 Trend vs Other Diagnostics
Linear trend properties compared to scatter, variability, and coverage.

In [ ]:
scatter_grid([
    ('stats_trend_slope_mag_per_year', 'stats_photometry_robust_sigma_mag'),
    ('stats_trend_slope_mag_per_year', 'stats_variability_reduced_chi2_vs_constant'),
    ('stats_trend_r2',                 'stats_variability_reduced_chi2_vs_constant'),
    ('stats_trend_r2',                 'stats_photometry_robust_sigma_mag'),
    ('stats_trend_slope_mag_per_year', 'stats_time_span_days'),
    ('stats_trend_r2',                 'stats_file_points_kept_after_filter'),
], title='Trend vs Other Diagnostics',
   log_axes={'stats_variability_reduced_chi2_vs_constant',
             'stats_file_points_kept_after_filter'})

### 8f \u2013 Compact Pair Plot (top variability stats)
Seaborn pairplot of 5 key variability/photometry stats for a compact overview.

In [ ]:
PAIR_COLS = [
    'stats_photometry_robust_sigma_mag',
    'stats_variability_reduced_chi2_vs_constant',
    'stats_variability_von_neumann_ratio',
    'stats_variability_stetson_J',
    'stats_variability_stetson_K',
]
PAIR_COLS = [c for c in PAIR_COLS if c in STATS_COLS]

pair_df = df_all[['mag_bin'] + PAIR_COLS].dropna().copy()
# log-transform chi2 for readability
chi2_col = 'stats_variability_reduced_chi2_vs_constant'
if chi2_col in pair_df.columns:
    pair_df = pair_df[pair_df[chi2_col] > 0].copy()
    pair_df[chi2_col] = np.log10(pair_df[chi2_col])

# clip to 1st-99th pctl per column
for c in PAIR_COLS:
    lo, hi = pair_df[c].quantile([0.01, 0.99])
    pair_df = pair_df[(pair_df[c] >= lo) & (pair_df[c] <= hi)]

short_rename = {c: c.removeprefix('stats_') for c in PAIR_COLS}
if chi2_col in short_rename:
    short_rename[chi2_col] = 'log10(reduced_chi2)'
pair_df = pair_df.rename(columns=short_rename)

present_labels = ordered_mag_bins(pair_df['mag_bin'])
pair_palette = {label: BIN_PALETTE[label] for label in present_labels}
g = sns.pairplot(pair_df, hue='mag_bin', hue_order=present_labels, palette=pair_palette,
                 plot_kws={'s': 5, 'alpha': 0.15, 'edgecolor': 'none',
                           'rasterized': True},
                 diag_kws={'alpha': 0.5},
                 height=2.2, aspect=1.0,
                 corner=True)
g.figure.suptitle('Pair Plot: Key Variability Stats', y=1.01,
                  fontsize=14, fontweight='bold')
plt.show()


## \u00a79 \u2013 Outlier Flagging
Candidates whose variability stats fall **>3 MAD** from the per-mag-bin median.

In [ ]:
OUTLIER_COLS = [
    'stats_variability_reduced_chi2_vs_constant',
    'stats_variability_von_neumann_ratio',
    'stats_variability_lag1_autocorr',
    'stats_variability_stetson_J',
    'stats_variability_stetson_K',
    'stats_photometry_robust_sigma_mag',
    'stats_trend_slope_mag_per_year',
]
OUTLIER_COLS = [c for c in OUTLIER_COLS if c in STATS_COLS]

outlier_flags = pd.DataFrame(index=df_all.index)

for col in OUTLIER_COLS:
    med = df_all.groupby('mag_bin')[col].transform('median')
    mad = df_all.groupby('mag_bin')[col].transform(
        lambda s: np.median(np.abs(s - s.median())))
    mad = mad.replace(0, np.nan)
    deviation = np.abs(df_all[col] - med) / (1.4826 * mad)
    outlier_flags[col] = deviation > 3.0

df_all['n_outlier_stats'] = outlier_flags.sum(axis=1)
df_outliers = df_all[df_all['n_outlier_stats'] >= 2].copy()

display_cols = ['mag_bin', 'path', 'n_outlier_stats'] + OUTLIER_COLS
print(f'Candidates flagged as outlier on >=2 stats: '
      f'{len(df_outliers):,} / {len(df_all):,}')
df_outliers[display_cols].sort_values(
    'n_outlier_stats', ascending=False).head(30)

In [ ]:
# ── outlier count per mag bin ─────────────────────────────────────
outlier_summary = (
    df_all.groupby('mag_bin')['n_outlier_stats']
    .agg(total='size',
         outliers_ge2=lambda s: (s >= 2).sum(),
         outliers_ge3=lambda s: (s >= 3).sum())
)
outlier_summary['pct_ge2'] = (
    100 * outlier_summary['outliers_ge2'] / outlier_summary['total']).round(1)
outlier_summary['pct_ge3'] = (
    100 * outlier_summary['outliers_ge3'] / outlier_summary['total']).round(1)
outlier_summary